# 공유오피스 결제 전환 예측 — 모델링

**목표**: 무료체험 유저의 결제 전환을 예측해, 결제가 애매한 유저에게 프로모션 메일을 보낸다.
→ 메일은 비용이 낮아 **놓치지 않는 것(Recall)** 이 중요하므로 **F2 score**를 주지표로 본다.

**흐름**
1. 데이터 불러오기 · 확인
2. **사전 실험** (no-show 제외 / 불균형 보정 / 요일 제외) → 전처리 방침 결정
3. 결정 사항 반영 → 본 모델 학습 (5개 모델, 공정 비교)
4. 결과 비교
5. Optuna 튜닝
6. SHAP 해석

> **사전 실험으로 내린 결론** (2장에서 검증):
> - **no-show 유저 제외**: 변수가 없어(요일 외 전부 0/결측) 예측 불가 → 모델 학습에서 제외
> - **클래스 불균형 보정 적용**: F2(Recall) 개선을 위해 전 모델에 class_weight 적용
> - **요일(signup_dow)**: 약한 변수지만 제외 시 AUC가 하락해 본 학습에서 유지

## 1. 라이브러리 & 데이터 불러오기

In [ ]:
import warnings
warnings.filterwarnings('ignore')   # 경고 메시지 숨기기

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import fbeta_score, accuracy_score, roc_auc_score, f1_score, precision_score, recall_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

!pip install catboost
from catboost import CatBoostClassifier

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform, os

system = platform.system()
if system == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif system == 'Darwin':  # Mac
    plt.rc('font', family='AppleGothic')
else:  # Linux / Colab
    # 1) 나눔고딕 설치 (이미 있으면 빠르게 넘어감)
    os.system('apt-get -qq -y install fonts-nanum > /dev/null')
    # 2) 설치된 폰트를 matplotlib에 등록
    font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    fm.fontManager.addfont(font_path)          # 폰트 직접 등록
    plt.rc('font', family='NanumGothic')       

plt.rcParams['axes.unicode_minus'] = False
print('폰트 설정:', plt.rcParams['font.family'])

# 테스트
plt.title("한글 깨짐 테스트 완료!")
plt.plot([1, 2, 3], [4, 5, 6])
plt.show()

In [ ]:
# from google.colab import drive   # Colab 전용
# drive.mount('/content/drive')

In [ ]:
df_raw = pd.read_csv("data/df_master_output.csv")
df = df_raw.copy()
df

### 데이터 확인

In [ ]:
# 타겟(결제 여부) 비율 — 약 38%가 결제(1) → 약간의 불균형
print('타겟 분포:')
print(df['is_payment'].value_counts(normalize=True).round(3))

# 결측치 현황 — 세 컬럼에 결측이 있고, 모두 'no-show(미방문)' 유저에게서 발생
print('\n결측치 현황:')
print(df.isna().sum()[df.isna().sum() > 0])

## 2. 사전 실험 — 전처리 방침 결정

본 학습에 들어가기 전, 세 가지를 **직접 돌려서** 확인하고 방침을 정한다.
실험은 빠르고 결측을 자동 처리하는 **XGBoost**로 통일한다.

In [ ]:
target = 'is_payment'
cat_cols    = ['primary_site', 'signup_dow']                  # 범주형
num_with_na = ['primary_site_area', 'last_visit_to_end_gap']  # 결측 있는 수치형

from xgboost import XGBClassifier
from sklearn.metrics import fbeta_score, f1_score

# 실험용 공통 전처리 함수: primary_site 결측→'none', 범주형 원-핫
def split_and_align(data, drop_cols=None):
    d = data.copy()
    if drop_cols:
        d = d.drop(columns=drop_cols)
    d['primary_site'] = d['primary_site'].fillna('none').astype(str)
    use_cat = [c for c in cat_cols if c in d.columns]
    Xx = pd.get_dummies(d.drop(columns=target), columns=use_cat)
    yy = d[target]
    Xtr, Xte, ytr, yte = train_test_split(
        Xx, yy, test_size=0.2, random_state=42, stratify=yy)
    Xtr, Xte = Xtr.align(Xte, join='left', axis=1, fill_value=0)
    return Xtr, Xte, ytr, yte

### 2-1. 실험 ① no-show 유저 제외 vs 유지

no-show(`visit_days==0`) 유저는 가입 요일 외 모든 값이 0/결측이라 예측 근거가 거의 없다.
이들이 성능에 방해가 되는지 확인한다.

In [ ]:
exp1 = []

# (A) no-show 유지 (전체)
Xtr_k, Xte_k, ytr_k, yte_k = split_and_align(df)
m_k = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                    eval_metric='logloss', random_state=42).fit(Xtr_k, ytr_k)
exp1.append({'시나리오': 'no-show 유지(전체)', 'n': len(df),
             'AUC': round(roc_auc_score(yte_k, m_k.predict_proba(Xte_k)[:, 1]), 4)})

# (B) no-show 제외 (방문 유저만)
df_visited = df[df['visit_days'] > 0]
Xtr_d, Xte_d, ytr_d, yte_d = split_and_align(df_visited)
m_d = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                    eval_metric='logloss', random_state=42).fit(Xtr_d, ytr_d)
exp1.append({'시나리오': 'no-show 제외(방문자만)', 'n': len(df_visited),
             'AUC': round(roc_auc_score(yte_d, m_d.predict_proba(Xte_d)[:, 1]), 4)})

pd.DataFrame(exp1)

**결론**: no-show를 제외하면 AUC가 오른다. 이들은 변수가 없어 모델이 학습할 게 없고,
'애매한 유저(메일 타겟)'를 가려낼 근거도 없다. → **본 학습에서 no-show를 제외**한다.
(단 버리는 게 아니라, 실제 운영 시 no-show는 평균 전환율을 적용하는 별도 규칙으로 처리.)

### 2-2. 실험 ② 불균형 보정(class_weight) 적용 vs 미적용

전환율 약 38%의 불균형. 메일 프로모션은 **Recall(놓치지 않기)** 이 중요하므로,
보정이 F2/Recall을 올리는지 확인한다. (이 실험은 no-show 제외 데이터 기준)

In [ ]:
Xtr_w, Xte_w, ytr_w, yte_w = split_and_align(df_visited)
spw = (ytr_w == 0).sum() / (ytr_w == 1).sum()   # 미결제/결제 비율
print(f'scale_pos_weight = {spw:.3f}')

exp2 = []
for label, kwargs in [('미적용', {}), ('적용', {'scale_pos_weight': spw})]:
    m = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                      eval_metric='logloss', random_state=42, **kwargs).fit(Xtr_w, ytr_w)
    pred = m.predict(Xte_w); proba = m.predict_proba(Xte_w)[:, 1]
    exp2.append({'보정': label,
                 'AUC': round(roc_auc_score(yte_w, proba), 4),
                 'F2' : round(fbeta_score(yte_w, pred, beta=2), 4)})
pd.DataFrame(exp2)

**결론**: 보정은 AUC는 거의 안 바꾸지만 **F2(Recall)를 크게 올린다**.
메일 프로모션 목적(놓치지 않기)에 맞으므로 → **본 학습에서 전 모델에 불균형 보정 적용**.

### 2-3. 실험 ③ 가입 요일(signup_dow) 포함 vs 제외

요일은 EDA에서 전환율 차이가 거의 없었다(36~40%). 약한 변수가 노이즈인지 확인한다.

In [ ]:
exp3 = []
# 포함
Xtr_a, Xte_a, ytr_a, yte_a = split_and_align(df_visited)
m_a = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                    eval_metric='logloss', random_state=42).fit(Xtr_a, ytr_a)
exp3.append({'시나리오': '요일 포함',
             'AUC': round(roc_auc_score(yte_a, m_a.predict_proba(Xte_a)[:, 1]), 4)})
# 제외
Xtr_b, Xte_b, ytr_b, yte_b = split_and_align(df_visited, drop_cols=['signup_dow'])
m_b = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                    eval_metric='logloss', random_state=42).fit(Xtr_b, ytr_b)
exp3.append({'시나리오': '요일 제외',
             'AUC': round(roc_auc_score(yte_b, m_b.predict_proba(Xte_b)[:, 1]), 4)})
pd.DataFrame(exp3)

**결론**: 요일을 빼면 AUC가 0.611 → 0.595로 하락한다.
EDA상 요일별 전환율 차이는 작았지만(36~40%) 다른 변수와 결합해 일부 기여하는 것으로 보고, 본 학습에서 유지한다.

---
### 📌 사전 실험 종합 → 본 학습 방침
- **no-show 제외** (변수 없음, 예측 불가)
- **전 모델 불균형 보정 적용** (F2/Recall 개선)
- 평가 주지표: **F2** (+ AUC 보조)

## 3. 본 모델 학습 (방침 반영)

사전 실험 결론대로 **no-show 제외 데이터**에 **불균형 보정**을 적용해 5개 모델을 학습한다.
모델마다 전처리(인코딩·스케일링·결측)는 특성에 맞게 다르게 적용한다.

### 3-1. no-show 제외 & train/test 분리

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# no-show 제외 (사전 실험 결론)
df_model = df[df['visit_days'] > 0].reset_index(drop=True)
print(f'no-show 제외: {len(df)} → {len(df_model)} 명')

X = df_model.drop(columns=target)
y = df_model[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 불균형 보정용 가중치 (train 기준)
spw = (y_train == 0).sum() / (y_train == 1).sum()
print(f'train: {X_train.shape} | scale_pos_weight = {spw:.3f}')

### 3-2. 평가 함수
F2를 주지표로, AUC·Recall 등을 함께 본다.

In [ ]:
results = []
def evaluate(name, model, X_te):
    pred  = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    acc  = accuracy_score(y_test, pred)
    auc  = roc_auc_score(y_test, proba)
    f1   = f1_score(y_test, pred)
    f2   = fbeta_score(y_test, pred, beta=2)   # 주지표
    prec = precision_score(y_test, pred)
    rec  = recall_score(y_test, pred)
    print(f'[{name}] F2={f2:.3f} | AUC={auc:.3f} | Recall={rec:.3f} | Prec={prec:.3f} | F1={f1:.3f} | Acc={acc:.3f}')
    results.append({'model': name, 'f2': round(f2,3), 'auc': round(auc,3),
                    'recall': round(rec,3), 'precision': round(prec,3)})

# 스케일링할 수치형 컬럼 (이진/원-핫 제외)
num_to_scale = ['stay_total_sec', 'stay_mean_sec', 'visit_days', 'n_access_events',
                'n_sites_visited', 'primary_site_area', 'last_visit_to_end_gap']

### 3-3. 모델 ① Logistic Regression
결측 채움 + 원-핫 + (수치형만)스케일링 + `class_weight='balanced'`

In [ ]:
tr_lr = X_train.copy(); te_lr = X_test.copy()

# primary_site 결측 → 'none'
tr_lr['primary_site'] = tr_lr['primary_site'].fillna('none').astype(str)
te_lr['primary_site'] = te_lr['primary_site'].fillna('none').astype(str)

# 수치형 결측 → 중앙값(train 기준) + 미방문 플래그
for c in num_with_na:
    median = tr_lr[c].median()
    tr_lr[c + '_missing'] = tr_lr[c].isna().astype(int)
    te_lr[c + '_missing'] = te_lr[c].isna().astype(int)
    tr_lr[c] = tr_lr[c].fillna(median); te_lr[c] = te_lr[c].fillna(median)

# 원-핫
tr_lr = pd.get_dummies(tr_lr, columns=cat_cols)
te_lr = pd.get_dummies(te_lr, columns=cat_cols)
tr_lr, te_lr = tr_lr.align(te_lr, join='left', axis=1, fill_value=0)

# 수치형만 스케일링 (원-핫 0/1은 그대로)
scaler_lr = StandardScaler()
tr_lr[num_to_scale] = scaler_lr.fit_transform(tr_lr[num_to_scale])
te_lr[num_to_scale] = scaler_lr.transform(te_lr[num_to_scale])

logreg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
logreg.fit(tr_lr, y_train)
evaluate('LogisticRegression', logreg, te_lr)

### 3-4. 모델 ② Decision Tree
결측 -1 + 원-핫 + `class_weight='balanced'` (스케일링 불필요)

In [ ]:
tr_dt = X_train.copy(); te_dt = X_test.copy()
tr_dt['primary_site'] = tr_dt['primary_site'].fillna('none').astype(str)
te_dt['primary_site'] = te_dt['primary_site'].fillna('none').astype(str)
for c in num_with_na:
    tr_dt[c] = tr_dt[c].fillna(-1); te_dt[c] = te_dt[c].fillna(-1)
tr_dt = pd.get_dummies(tr_dt, columns=cat_cols)
te_dt = pd.get_dummies(te_dt, columns=cat_cols)
tr_dt, te_dt = tr_dt.align(te_dt, join='left', axis=1, fill_value=0)

dtree = DecisionTreeClassifier(max_depth=5, min_samples_leaf=20,
                               class_weight='balanced', random_state=42)
dtree.fit(tr_dt, y_train)
evaluate('DecisionTree', dtree, te_dt)

### 3-5. 모델 ③ XGBoost
원-핫 + 수치 결측 NaN 그대로 + `scale_pos_weight`

In [ ]:
tr_xgb = X_train.copy(); te_xgb = X_test.copy()
tr_xgb['primary_site'] = tr_xgb['primary_site'].fillna('none').astype(str)
te_xgb['primary_site'] = te_xgb['primary_site'].fillna('none').astype(str)
tr_xgb = pd.get_dummies(tr_xgb, columns=cat_cols)
te_xgb = pd.get_dummies(te_xgb, columns=cat_cols)
tr_xgb, te_xgb = tr_xgb.align(te_xgb, join='left', axis=1, fill_value=0)

xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                    scale_pos_weight=spw, eval_metric='logloss', random_state=42)
xgb.fit(tr_xgb, y_train)
evaluate('XGBoost', xgb, te_xgb)

### 3-6. 모델 ④ LightGBM
XGBoost와 동일 전처리 + `class_weight='balanced'`

In [ ]:
tr_lgbm = X_train.copy(); te_lgbm = X_test.copy()
tr_lgbm['primary_site'] = tr_lgbm['primary_site'].fillna('none').astype(str)
te_lgbm['primary_site'] = te_lgbm['primary_site'].fillna('none').astype(str)
tr_lgbm = pd.get_dummies(tr_lgbm, columns=cat_cols)
te_lgbm = pd.get_dummies(te_lgbm, columns=cat_cols)
tr_lgbm, te_lgbm = tr_lgbm.align(te_lgbm, join='left', axis=1, fill_value=0)

lgbm = LGBMClassifier(n_estimators=300, learning_rate=0.1,
                      class_weight='balanced', random_state=42, verbose=-1)
lgbm.fit(tr_lgbm, y_train)
evaluate('LightGBM', lgbm, te_lgbm)

### 3-7. 모델 ⑤ CatBoost
범주형 직접 처리(`cat_features`) + `auto_class_weights='Balanced'`

In [ ]:
tr_cat = X_train.copy(); te_cat = X_test.copy()
tr_cat['primary_site'] = tr_cat['primary_site'].fillna('none').astype(str)
te_cat['primary_site'] = te_cat['primary_site'].fillna('none').astype(str)
tr_cat['signup_dow'] = tr_cat['signup_dow'].astype(str)
te_cat['signup_dow'] = te_cat['signup_dow'].astype(str)

cat = CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1,
                         auto_class_weights='Balanced',
                         cat_features=['primary_site', 'signup_dow'],
                         verbose=0, random_state=42)
cat.fit(tr_cat, y_train)
evaluate('CatBoost', cat, te_cat)

### 공통 함수: 혼동행렬 시각화

혼동행렬(Confusion Matrix)은 모델이 **무엇을 맞히고 무엇을 틀렸는지**를 4칸으로 보여준다.
- 좌상(TN): 미결제를 미결제로 (정답)
- 우하(TP): 결제를 결제로 (정답)
- 우상(FP): 미결제인데 결제로 잘못 예측
- 좌하(FN): 결제인데 미결제로 놓침 ← **메일 프로모션에서 줄이고 싶은 것**

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def show_confusion(model, X_te, title):
    """모델의 test 예측으로 혼동행렬을 그린다."""
    pred = model.predict(X_te)
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['미결제(0)', '결제(1)'])
    disp.plot(cmap='Blues', values_format='d')
    plt.title(title)
    plt.tight_layout()
    plt.show()
    return cm

## 4. 결과 비교
**F2 기준**으로 정렬한다. (모든 모델에 불균형 보정 적용 → 공정 비교)

In [ ]:
result_df = pd.DataFrame(results).sort_values('f2', ascending=False).reset_index(drop=True)
result_df

### 4-1. 5개 모델 혼동행렬

각 모델이 결제자(1)를 얼마나 잡아내고(TP), 얼마나 놓치는지(FN) 비교한다.

In [ ]:
# 5개 모델을 (이름, 모델, test데이터)로 묶어 반복
model_list = [
    ('LogisticRegression', logreg, te_lr),
    ('DecisionTree',       dtree,  te_dt),
    ('XGBoost',            xgb,    te_xgb),
    ('LightGBM',           lgbm,   te_lgbm),
    ('CatBoost',           cat,    te_cat),
]
for name, model, X_te in model_list:
    show_confusion(model, X_te, f'혼동행렬 — {name}')

## 5. 하이퍼파라미터 튜닝 (Optuna) — 지표별 3가지 버전

같은 모델(XGBoost)이라도 **무엇을 목표로 튜닝하느냐**에 따라 결과가 달라진다.
세 가지 지표를 각각 목표로 튜닝해 비교한다.

- **5-1. 정확도(Accuracy) 최적화** — 전체를 많이 맞히는 데 집중
- **5-2. F2 최적화** — Recall 가중. **우리 프로모션 목적(놓치지 않기)에 맞는 주지표**
- **5-3. Recall 최적화** — 결제자를 최대한 놓치지 않는 데 집중

> 세 버전을 비교하면, "지표를 바꾸면 모델이 어떻게 달라지는가"를 직접 볼 수 있다.
> 메일 프로모션 목적상 최종적으로는 **F2 버전**을 채택한다.

In [ ]:
!pip install optuna -q

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, fbeta_score

# XGBoost 전처리 데이터 재사용 (tr_xgb, te_xgb)
# 지표(scoring)만 바꿔서 재사용할 objective 생성 함수
def make_objective(scoring):
    def objective(trial):
        params = {
            'n_estimators' : trial.suggest_int('n_estimators', 100, 500, step=50),
            'max_depth'    : trial.suggest_int('max_depth', 2, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
            'subsample'    : trial.suggest_float('subsample', 0.6, 1.0),
        }
        model = XGBClassifier(**params, scale_pos_weight=spw,
                              eval_metric='logloss', random_state=42)
        # train 안에서 3-fold 교차검증 (test는 안 봄 → 누수 방지)
        return cross_val_score(model, tr_xgb, y_train, cv=3, scoring=scoring).mean()
    return objective

# F2용 scorer (beta=2)
f2_scorer = make_scorer(fbeta_score, beta=2)

# 결과를 모아둘 리스트
tuning_results = []

### 5-1. 정확도(Accuracy) 최적화

In [ ]:
study_acc = optuna.create_study(direction='maximize')
study_acc.optimize(make_objective('accuracy'), n_trials=100)

xgb_acc = XGBClassifier(**study_acc.best_params, scale_pos_weight=spw,
                        eval_metric='logloss', random_state=42).fit(tr_xgb, y_train)
print('[정확도 최적화] best params:', study_acc.best_params)
evaluate('XGBoost(정확도 최적화)', xgb_acc, te_xgb)

In [ ]:
# [정확도 최적화] Optuna 탐색 시각화
import optuna.visualization as vis
vis.plot_optimization_history(study_acc).show()   # 탐색 과정(점수 수렴)
vis.plot_param_importances(study_acc).show()       # 파라미터 중요도

In [ ]:
# [정확도 최적화] 혼동행렬
show_confusion(xgb_acc, te_xgb, '혼동행렬 — XGBoost (정확도 최적화)')

### 5-2. F2 최적화 ⭐ (주지표)

In [ ]:
study_f2 = optuna.create_study(direction='maximize')
study_f2.optimize(make_objective(f2_scorer), n_trials=100)

xgb_f2 = XGBClassifier(**study_f2.best_params, scale_pos_weight=spw,
                       eval_metric='logloss', random_state=42).fit(tr_xgb, y_train)
print('[F2 최적화] best params:', study_f2.best_params)
evaluate('XGBoost(F2 최적화)', xgb_f2, te_xgb)

In [ ]:
# [F2 최적화] Optuna 탐색 시각화
import optuna.visualization as vis
vis.plot_optimization_history(study_f2).show()   # 탐색 과정(점수 수렴)
vis.plot_param_importances(study_f2).show()       # 파라미터 중요도

In [ ]:
# [F2 최적화] 혼동행렬
show_confusion(xgb_f2, te_xgb, '혼동행렬 — XGBoost (F2 최적화)')

### 5-3. Recall 최적화

In [ ]:
study_rec = optuna.create_study(direction='maximize')
study_rec.optimize(make_objective('recall'), n_trials=100)

xgb_recall = XGBClassifier(**study_rec.best_params, scale_pos_weight=spw,
                           eval_metric='logloss', random_state=42).fit(tr_xgb, y_train)
print('[Recall 최적화] best params:', study_rec.best_params)
evaluate('XGBoost(Recall 최적화)', xgb_recall, te_xgb)

In [ ]:
# [Recall 최적화] Optuna 탐색 시각화
import optuna.visualization as vis
vis.plot_optimization_history(study_rec).show()   # 탐색 과정(점수 수렴)
vis.plot_param_importances(study_rec).show()       # 파라미터 중요도

In [ ]:
# [Recall 최적화] 혼동행렬
show_confusion(xgb_recall, te_xgb, '혼동행렬 — XGBoost (Recall 최적화)')

### 5-4. 세 최적화 버전 비교

같은 XGBoost인데 **튜닝 목표만 다르게** 한 세 버전을 비교한다.
각 버전이 자기가 최적화한 지표에서 강한지 확인한다.

In [ ]:
# 세 study의 CV best score를 dict로 매핑 (model 이름 ↔ best_value)
best_scores = {
    'XGBoost(정확도 최적화)': study_acc.best_value,
    'XGBoost(F2 최적화)'   : study_f2.best_value,
    'XGBoost(Recall 최적화)': study_rec.best_value,
}

# results 리스트에서 3개 최적화 버전만 추출
tuning_compare = pd.DataFrame([r for r in results if '최적화' in r['model']])
tuning_compare = tuning_compare.tail(3).reset_index(drop=True)

# CV best score 컬럼 추가 (각 모델이 자기 목표 지표에서 낸 train 교차검증 점수)
tuning_compare['cv_best_score'] = tuning_compare['model'].map(best_scores).round(3)

print('각 버전이 자기 목표 지표에서 강한지 확인:')
print('(cv_best_score = 각 study가 자기 목표로 최적화한 train 교차검증 점수)')
tuning_compare

**해석 포인트**
- **정확도 최적화** → Accuracy는 높지만 Recall/F2는 낮을 수 있다(소수 클래스를 덜 잡음).
- **Recall 최적화** → Recall은 최고지만 정확도/정밀도는 희생될 수 있다(많이 1로 예측).
- **F2 최적화** → 둘의 절충. **메일 프로모션 목적(놓치지 않되 과하지 않게)에 가장 적합.**
- → 최종 모델로 **F2 최적화 버전(`xgb_f2`)** 을 채택한다.

## 6. SHAP 해석 (F2 최적화 XGBoost)

어떤 변수가 결제 전환을 미는지 해석한다.

In [ ]:
import shap
explainer = shap.TreeExplainer(xgb_f2)
shap_values = explainer(te_xgb)
print('SHAP shape:', shap_values.shape)

### 6-1. 전체 변수 중요도 (bar)

In [ ]:
import matplotlib.pyplot as plt
shap.plots.bar(shap_values, show=False); plt.tight_layout(); plt.show()

### 6-2. 영향 분포 (beeswarm)

In [ ]:
shap.plots.beeswarm(shap_values, show=False); plt.tight_layout(); plt.show()

### 6-3. 개별 유저 예측 (waterfall)

In [ ]:
shap.plots.waterfall(shap_values[0], show=False); plt.tight_layout(); plt.show()

## 7. 피처 선택 실험 — SHAP 상위 5개 vs 전체

SHAP 중요도(6장)에서 상위 5개 피처가 대부분의 기여를 차지했다:
`stay_mean_sec`, `last_visit_to_end_gap`, `visit_days`, `stay_total_sec`, `n_access_events`.

기여가 적은 나머지 피처를 빼고 **상위 5개만**으로 학습하면 성능이 어떻게 변하는지 확인한다.
(no-show 제외 · 불균형 보정은 그대로 유지, 여러 지표로 비교)

In [ ]:
# SHAP 상위 5개 피처 (전부 수치형 → 원-핫 불필요)
top5_features = ['stay_mean_sec', 'last_visit_to_end_gap', 'visit_days',
                 'stay_total_sec', 'n_access_events']

# 공통 평가 함수 (여러 지표 한 번에)
def eval_metrics(model, X_te, y_te):
    pred  = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    return {
        'AUC'      : round(roc_auc_score(y_te, proba), 3),
        'F2'       : round(fbeta_score(y_te, pred, beta=2), 3),
        'F1'       : round(f1_score(y_te, pred), 3),
        'Recall'   : round(recall_score(y_te, pred), 3),
        'Precision': round(precision_score(y_te, pred), 3),
        'Accuracy' : round(accuracy_score(y_te, pred), 3),
    }

### 7-1. 전체 피처 버전 (기준)

In [ ]:
# 전체 피처: 앞서 만든 XGBoost 전처리(tr_xgb, te_xgb) 재사용
# (no-show 제외 + 불균형 보정 spw 포함)
feat_results = []

xgb_full = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                         scale_pos_weight=spw, eval_metric='logloss',
                         random_state=42).fit(tr_xgb, y_train)
m_full = eval_metrics(xgb_full, te_xgb, y_test)
m_full['피처셋'] = f'전체 ({tr_xgb.shape[1]}개)'
feat_results.append(m_full)
print('전체 피처:', m_full)

### 7-2. 상위 5개 피처 버전

In [ ]:
# 상위 5개만 추출 (수치형이라 원-핫 불필요, 결측은 XGBoost가 자동 처리)
tr_top5 = X_train[top5_features].copy()
te_top5 = X_test[top5_features].copy()

xgb_top5 = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                         scale_pos_weight=spw, eval_metric='logloss',
                         random_state=42).fit(tr_top5, y_train)
m_top5 = eval_metrics(xgb_top5, te_top5, y_test)
m_top5['피처셋'] = '상위 5개'
feat_results.append(m_top5)
print('상위 5개:', m_top5)

### 7-3. 비교

In [ ]:
compare_feat = pd.DataFrame(feat_results)[
    ['피처셋', 'AUC', 'F2', 'F1', 'Recall', 'Precision', 'Accuracy']
]
compare_feat

**해석 포인트**
- 상위 5개로 줄이면 **전 지표가 하락**했다 (AUC 0.616 → 0.591, F2 0.560 → 0.539, Recall 0.585 → 0.559).
- SHAP 기여도가 작아 보이는 나머지 피처도 **예측에 일부 기여**하고 있었다는 뜻이다.
- 모델 단순화 이득보다 성능 손실이 커서, **최종 모델은 전체 피처를 유지**한다.

> ⚠️ 단일 split 결과이므로, 더 엄밀히는 **교차검증**으로 확인하는 것이 좋다.

---
**정리**
- 사전 실험으로 **no-show 제외 + 불균형 보정**을 결정하고 본 학습에 반영했다.
- 5개 모델을 동일 조건(보정 적용)으로 공정 비교, **F2** 기준 선정.
- Optuna로 F2 튜닝 후 SHAP으로 해석.
- 다음: 메일 타겟팅(확률 중간대 추출), Streamlit 대시보드로 확장.